__This will be a walkthrough of uploading sample data to the data management system for each of the five task types.__

1. Single Label Classification
2. Muli-Label Classification
3. Object Detection
4. Semantic Segmentation
5. Instance Segmentation

Note, these are not training samples; they are examples on how to ingest data into the infrastructure described in the README. It references the expected input data format (consistent with Ground Truth output) via the manifests in the samples folder and allows you to test the infrastructure for different label types. While I did create a logging client that returns logs in pandas DataFrame format (and include an example), I highly recommend simply using Athena in the AWS console for log and table visualization. The same holds true for the step function flow to detect any failure points.

We must login to our account and gain credentials and permissions to interact with the infrastructure. Set up a user in Identity Center assigned to the account containing the CDK app, grant necessary permissions, and login via the terminal to gain temporary credentials. Ensure the profile you are using is in the local AWS config file and points to the appropriate account, role, and AWS region.

In [35]:
# This is the name of the CDK app you deployed.
app_name = "cvdmsv1"

# This is your profile name (see the local aws config file).
profile_name = "developers_admin"

# Login to AWS. This will redirect you to a login screen or be approved if credentials are still valid from your previous login.
!aws sso login --profile {profile_name}

Attempting to automatically open the SSO authorization page in your default browser.
If the browser does not open, open the following URL:

https://oidc.us-east-1.amazonaws.com/authorize?response_type=code&client_id=BdXVxybpAKILVt2_MS6XoXVzLWVhc3QtMQ&redirect_uri=http%3A%2F%2F127.0.0.1%3A51529%2Foauth%2Fcallback&state=0053857b-e49d-4b6c-95fb-54d536d04c10&code_challenge_method=S256&scopes=sso%3Aaccount%3Aaccess&code_challenge=rQJSXJxCgap05O8eO1Uf4awXKoYPIlUXrmhvCsoaJlY
Successfully logged into Start URL: https://d-906625d369.awsapps.com/start


The following imports the programmatic API main entry point for interacting with the app. Everything is accessible through this client.

In [36]:
# Instantiate the client.
from cvdms_platform import CvdmsApp

app = CvdmsApp(app_name=app_name,
               profile_name=profile_name)

2025-12-29 21:59:26,297 - INFO - Instantiating user instance.
2025-12-29 21:59:26,355 - INFO - Loading cached SSO token for myDefaultSession
2025-12-29 21:59:28,644 - INFO - Using AWS profile: developers_admin, user = developer_brian, region: us-east-1, passed config loading step.
2025-12-29 21:59:28,712 - INFO - Instantiation complete.


This initiates the upload workflow for a manifest of images and labels.

In [37]:
manifest_path = r"samples/single_label/single_label_truth_output.manifest"
label_type = "single-label"
job_summary = "Upload some sample COCO images for single label label type."
data_source = "cocoval2017"

upload_info = app.start_upload_job(manifest_path,
                                   label_type,
                                   job_summary = job_summary,
                                   data_source = data_source)

2025-12-29 21:59:33,688 - INFO - Acquiring lock for lock global, new potential holder = 88c27e56-3ecc-4d0f-80d3-20109058d94d, used lock table name cvdmsv1-StorageStack-LockTableB9DACF42-13ZZ8FZ970ZGZ
2025-12-29 21:59:34,191 - INFO - Acquired lock: 88c27e56-3ecc-4d0f-80d3-20109058d94d
2025-12-29 21:59:34,260 - INFO - Created job row in job table for IMAGE_UPLOAD event and is status: PENDING.
2025-12-29 21:59:34,262 - INFO - Manifest validated. Uploading to S3...
2025-12-29 21:59:34,827 - INFO - Upload of manifest success: s3://cvdmsv1-storagestack-s3filebucketab18cd0f-jmwlzpv2lzhx/temp/image-upload/88c27e56-3ecc-4d0f-80d3-20109058d94d/88c27e56-3ecc-4d0f-80d3-20109058d94d.manifest
2025-12-29 21:59:34,972 - INFO - Upload of job.json success: s3://cvdmsv1-storagestack-s3filebucketab18cd0f-jmwlzpv2lzhx/temp/image-upload/88c27e56-3ecc-4d0f-80d3-20109058d94d/job.json
2025-12-29 21:59:34,973 - INFO - Done uploading manifest and job.json to S3.
2025-12-29 21:59:35,036 - INFO - Successfully upda

Run this cell to verify the job was started successfully and retrieve the job id.

In [39]:
job_id = upload_info.get('job_id')
upload_error = upload_info.get('error')

if job_id:
    print(f"Upload start was a success, job id is {job_id}")
else:
    print(f"Upload start was a failure, error is {upload_error}, see log file in cvdms_platform/api_logs")

Upload start was a success, job id is 88c27e56-3ecc-4d0f-80d3-20109058d94d


Use the log interface to see the logs for this job id stored in a pandas DataFrame. It can take some time for logs to show up.

In [ ]:
log_info = app.get_logs_by_job_id(job_id)

if not log_info.get('error'):
    log_df = log_info['logs_df']
    print('First 10 logs are:')
    print(log_df.head(10))
else:
    log_retrieval_error = log_info.get('error')
    print(f'Error getting logs: {log_retrieval_error}')